In case you dont want to compute the fine-tuned model you can load the model from local artifacts:

In [1]:
# Load fine-tuned model
import os, json, joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SAVE_DIR = "artifacts/jobbert_dept_finetuned"

if os.path.isdir(SAVE_DIR) and os.path.isfile(f"{SAVE_DIR}/config.json"):
    print("✅ Loading fine-tuned model from local artifacts...")
    tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR, fix_mistral_regex=True)
    model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)

    le = joblib.load(f"{SAVE_DIR}/label_encoder.joblib")
    meta = json.load(open(f"{SAVE_DIR}/meta.json"))
else:
    print("⚠️ No saved model found. Training from scratch...")
    # ... fine-tuning code ...


⚠️ No saved model found. Training from scratch...


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score

import torch


In [3]:
#Load Data
# Department dataset (labeled)
DEPARTMENT_DATA_PATH = "https://raw.githubusercontent.com/E-tech-coder/DataScienceCapstoneProject/refs/heads/main/department.csv"
df_department = pd.read_csv(DEPARTMENT_DATA_PATH)

# Profiles dataset (your unlabeled or partially labeled inference set)
PROFILES_PATH = "https://raw.githubusercontent.com/E-tech-coder/DataScienceCapstoneProject/refs/heads/main/df_profiles_cleansed.csv"
df_profiles = pd.read_csv(PROFILES_PATH)



In [4]:
#Standardize columns + clean
def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Standardize common naming differences
    if "position" not in df.columns and "text" in df.columns:
        df = df.rename(columns={"text": "position"})
    if "department" not in df.columns and "label" in df.columns:
        df = df.rename(columns={"label": "department"})

    return df

df_department = standardize_columns(df_department)
df_profiles   = standardize_columns(df_profiles)

# Basic cleaning
df_department = df_department.dropna(subset=["position", "department"]).copy()
df_department["position"] = df_department["position"].astype(str).str.strip()
df_department["department"] = df_department["department"].astype(str).str.strip()

df_profiles = df_profiles.dropna(subset=["position"]).copy()
df_profiles["position"] = df_profiles["position"].astype(str).str.strip()

print("Department rows:", len(df_department))
print("Profiles rows:", len(df_profiles))
print(df_department["department"].value_counts().head(10))


Department rows: 10145
Profiles rows: 2615
department
Marketing                 4295
Sales                     3328
Information Technology    1305
Business Development       620
Project Management         201
Consulting                 167
Administrative              83
Other                       42
Purchasing                  40
Customer Support            33
Name: count, dtype: int64


We build a prediction model based on approach 3 (Fine-tuned classification model) of the Assignment.
We employ JobBERT-V3 in our approach, as it is specifically designed for feature extraction and downstream supervised learning.

We chose JobBERT as the underlying language model because current research shows that domain-specific representations of job titles substantially outperform generic sentence encoders for HR-related prediction tasks.
Prior work demonstrates that JobBERT captures fine-grained semantic signals in job titles by leveraging large-scale co-occurrence information from vacancies and skills, making it particularly well suited for tasks such as job title normalization and organizational categorization (Decorte et al., 2021).
Consequently, JobBERT provides an optimal foundation for predicting departments based solely on job titles.


The contrastive pretraining  of JobBERT-V3 results in highly separable representations that are particularly well suited for fine-tuning classification models and for confidence-based decision thresholds in open-set inference settings.

[JobBERT-V3](https://https://huggingface.co/TechWolf/JobBERT-v3) is a domain-specific, contrastively trained model for job titles that delivers robust semantic representations across multiple languages (English, German, Spanish, and Chinese) without requiring task-specific supervision.
By combining large-scale multilingual training (21M+ job titles) with an efficiency-oriented architecture, it achieves state-of-the-art performance in both monolingual and cross-lingual job title matching

# 1) We start with a baseline (logistic regression) to check if our then fine tuned model performs better than classical machine learning models.

#1.1 JobBERT embeddings + Logistic Regression

In [5]:
#Train/test split on department.csv
le = LabelEncoder()
df_department["label_id"] = le.fit_transform(df_department["department"].values)

dept_train, dept_test = train_test_split(
    df_department,
    test_size=0.2,
    random_state=42,
    stratify=df_department["label_id"]
)

print("Train split:", len(dept_train), "Test split:", len(dept_test))
print("Num labels:", len(le.classes_))


Train split: 8116 Test split: 2029
Num labels: 11


In [6]:
#Embedding with SentenceTransformer JobBERT
!pip -q install sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

embedder = SentenceTransformer("TechWolf/JobBERT-v3")

X_train_text = dept_train["position"].tolist()
X_test_text  = dept_test["position"].tolist()

X_train_emb = embedder.encode(
    X_train_text,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)
X_test_emb = embedder.encode(
    X_test_text,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

y_train = dept_train["label_id"].values
y_test  = dept_test["label_id"].values


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/199 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

2_Asym/6235903824_Dense/model.safetensor(…):   0%|          | 0.00/3.15M [00:00<?, ?B/s]

2_Asym/6235904160_Dense/model.safetensor(…):   0%|          | 0.00/3.15M [00:00<?, ?B/s]

Batches:   0%|          | 0/254 [00:00<?, ?it/s]

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

In [7]:
#Train + evaluate
clf = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    n_jobs=-1
)
clf.fit(X_train_emb, y_train)

pred_test = clf.predict(X_test_emb)

print("=== BASELINE: Embeddings + LogisticRegression (Department Test) ===")
print("Accuracy:", accuracy_score(y_test, pred_test))
print("Macro F1:", f1_score(y_test, pred_test, average="macro"))
print(classification_report(y_test, pred_test, digits=3, target_names=le.classes_))


=== BASELINE: Embeddings + LogisticRegression (Department Test) ===
Accuracy: 0.9295219319862001
Macro F1: 0.8383296735008721
                        precision    recall  f1-score   support

        Administrative      0.519     0.824     0.636        17
  Business Development      0.776     0.919     0.841       124
            Consulting      0.780     0.970     0.865        33
      Customer Support      0.750     0.857     0.800         7
       Human Resources      0.600     1.000     0.750         6
Information Technology      0.890     0.897     0.893       261
             Marketing      0.984     0.936     0.959       859
                 Other      0.727     1.000     0.842         8
    Project Management      0.636     0.875     0.737        40
            Purchasing      0.889     1.000     0.941         8
                 Sales      0.975     0.938     0.956       666

              accuracy                          0.930      2029
             macro avg      0.775     0.

#1.2 Fine-tuning JobBERT on our Department Data

In the following we fine-tune JobBERT on our training data - the evaluation is conducted on our test data

In [8]:
#Install + prepare datasets
!pip -q install transformers datasets evaluate accelerate

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import evaluate

MODEL_NAME = "TechWolf/JobBERT-v3"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

num_labels = len(le.classes_)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [9]:
train_ds = Dataset.from_pandas(dept_train[["position", "label_id"]].rename(columns={"label_id": "labels"}))
test_ds  = Dataset.from_pandas(dept_test[["position", "label_id"]].rename(columns={"label_id": "labels"}))

def tokenize_fn(batch):
    return tokenizer(
        batch["position"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


Map:   0%|          | 0/8116 [00:00<?, ? examples/s]

Map:   0%|          | 0/2029 [00:00<?, ? examples/s]

In [10]:
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    out = acc_metric.compute(predictions=preds, references=labels)
    out["macro_f1"] = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return out


Fine Tune

In [11]:
model_ft = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

training_args = TrainingArguments(
    output_dir="jobbert_dept_finetuned",
    report_to="none",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True
)

trainer = Trainer(
    model=model_ft,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,   # we evaluate on the held-out department test split
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()
eval_out = trainer.evaluate()
eval_out


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at TechWolf/JobBERT-v3 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.149100,0.113438,0.989650,0.865284
2,0.068600,0.056399,0.995071,0.955872
3,0.041000,0.045996,0.997536,0.998205


{'eval_loss': 0.04599611833691597,
 'eval_accuracy': 0.9975357318876293,
 'eval_macro_f1': 0.9982051721210122,
 'eval_runtime': 5.9562,
 'eval_samples_per_second': 340.653,
 'eval_steps_per_second': 5.373,
 'epoch': 3.0}

In [12]:
#Save Fine-tuned model
SAVE_DIR = "artifacts/jobbert_dept_finetuned"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

import joblib, json, os
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(le, f"{SAVE_DIR}/label_encoder.joblib")
with open(f"{SAVE_DIR}/meta.json", "w") as f:
    json.dump(
        {"threshold": 0.55, "margin": 0.05, "labels": le.classes_.tolist()},
        f, indent=2
    )


In [13]:
#Test split classification report
pred_out = trainer.predict(test_ds)
logits = pred_out.predictions
pred_ids = logits.argmax(axis=1)

y_true_ids = dept_test["label_id"].values
y_pred_ids = pred_ids

y_true = le.inverse_transform(y_true_ids)
y_pred = le.inverse_transform(y_pred_ids)

print("=== FINETUNED JobBERT (Department Test) ===")
print(classification_report(y_true, y_pred, digits=3))


=== FINETUNED JobBERT (Department Test) ===
                        precision    recall  f1-score   support

        Administrative      1.000     1.000     1.000        17
  Business Development      0.992     0.992     0.992       124
            Consulting      1.000     1.000     1.000        33
      Customer Support      1.000     1.000     1.000         7
       Human Resources      1.000     1.000     1.000         6
Information Technology      0.992     0.989     0.990       261
             Marketing      1.000     0.999     0.999       859
                 Other      1.000     1.000     1.000         8
    Project Management      1.000     1.000     1.000        40
            Purchasing      1.000     1.000     1.000         8
                 Sales      0.997     1.000     0.999       666

              accuracy                          0.998      2029
             macro avg      0.998     0.998     0.998      2029
          weighted avg      0.998     0.998     0.998     

The results appear to be highly accurate - therefore we are checking with different measurements for data leakage

In [14]:
# (A) checking results:

print("dept_train rows:", len(dept_train))
print("dept_test rows:", len(dept_test))
print("train_ds rows:", len(train_ds))
print("test_ds rows:", len(test_ds))


dept_train rows: 8116
dept_test rows: 2029
train_ds rows: 8116
test_ds rows: 2029


In [15]:
#(B) Overlap of identical titles between train and test
train_titles = set(dept_train["position"].str.lower().str.strip())
test_titles  = set(dept_test["position"].str.lower().str.strip())
overlap = train_titles.intersection(test_titles)

print("Unique title overlap:", len(overlap))
print("Test titles seen in train (%):", len(overlap)/len(test_titles)*100)


Unique title overlap: 0
Test titles seen in train (%): 0.0


In [34]:
train_metrics = trainer.evaluate(train_ds)
test_metrics  = trainer.evaluate(test_ds)

print("Train metrics:", train_metrics)
print("Test metrics:", test_metrics)


Train metrics: {'eval_loss': 0.03373013436794281, 'eval_accuracy': 0.9995071463775259, 'eval_macro_f1': 0.9976285911352814, 'eval_runtime': 22.7161, 'eval_samples_per_second': 357.279, 'eval_steps_per_second': 5.591, 'epoch': 3.0}
Test metrics: {'eval_loss': 0.04599611833691597, 'eval_accuracy': 0.9975357318876293, 'eval_macro_f1': 0.9982051721210122, 'eval_runtime': 5.6618, 'eval_samples_per_second': 358.368, 'eval_steps_per_second': 5.652, 'epoch': 3.0}


In [30]:
import hashlib

def fingerprint_positions(df):
    s = "\n".join(df["position"].astype(str).head(200).tolist())
    return hashlib.md5(s.encode("utf-8")).hexdigest()

print("train fingerprint:", fingerprint_positions(dept_train))
print("test fingerprint :", fingerprint_positions(dept_test))


train fingerprint: 241c711805ef7e8459e34f89e3edb895
test fingerprint : d9b8baf4e371f2af9704890712d6d248


In [33]:
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
import numpy as np

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))


Accuracy: 0.7028985507246377
Balanced accuracy: 0.6879685247141766
Macro F1: 0.631922303804158


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


#2) Profiles inference + thresholding

We apply in the following our fine-tuned model on the profiles dataset

In [20]:
#Helper: predict probabilities on any dataframe
from datasets import Dataset

def predict_proba_for_positions(trainer: Trainer, df: pd.DataFrame, text_col="position"):
    ds = Dataset.from_pandas(df[[text_col]].copy())
    ds = ds.map(tokenize_fn, batched=True)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

    out = trainer.predict(ds)
    logits = out.predictions
    proba = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    return proba


In [21]:
#Predict on profiles
proba_p = predict_proba_for_positions(trainer, df_profiles, text_col="position")

pred_idx = proba_p.argmax(axis=1)
pred_conf = proba_p[np.arange(len(proba_p)), pred_idx]
pred_label = le.inverse_transform(pred_idx).astype(object)

df_profiles["pred_department_raw"] = pred_label
df_profiles["pred_conf"] = pred_conf

df_profiles[["position", "pred_department_raw", "pred_conf"]].head(10)


Map:   0%|          | 0/2615 [00:00<?, ? examples/s]

,position,pred_department_raw,pred_conf
0,Prokurist,Project Management,0.406388
1,CFO,Information Technology,0.678266
2,Betriebswirtin,Information Technology,0.555827
3,Prokuristin,Information Technology,0.408795
4,CFO,Information Technology,0.678266
5,Buchhalterin,Information Technology,0.628681
6,Solutions Architect,Information Technology,0.960333
7,Senior Network Engineer,Information Technology,0.909346
8,Manager of Network Services,Information Technology,0.935499
9,Infrastructure Administrator II,Information Technology,0.972277


In [22]:
#Threshold + top-2 margin
THRESH = 0.55
USE_MARGIN = True
MARGIN = 0.05

top2 = np.partition(proba_p, -2, axis=1)[:, -2]
margin = pred_conf - top2

pred_final = pred_label.copy()
pred_final[pred_conf < THRESH] = "Other"

if USE_MARGIN:
    unsure = (pred_final != "Other") & (margin < MARGIN)
    pred_final[unsure] = "Other"

df_profiles["pred_department"] = pred_final
df_profiles["pred_margin"] = margin

df_profiles[["position", "pred_department_raw", "pred_conf", "pred_margin", "pred_department"]].head(10)


,position,pred_department_raw,pred_conf,pred_margin,pred_department
0,Prokurist,Project Management,0.406388,0.134003,Other
1,CFO,Information Technology,0.678266,0.576670,Information Technology
2,Betriebswirtin,Information Technology,0.555827,0.420184,Information Technology
3,Prokuristin,Information Technology,0.408795,0.085635,Other
4,CFO,Information Technology,0.678266,0.576670,Information Technology
5,Buchhalterin,Information Technology,0.628681,0.465213,Information Technology
6,Solutions Architect,Information Technology,0.960333,0.952917,Information Technology
7,Senior Network Engineer,Information Technology,0.909346,0.875757,Information Technology
8,Manager of Network Services,Information Technology,0.935499,0.918349,Information Technology
9,Infrastructure Administrator II,Information Technology,0.972277,0.967681,Information Technology


In [23]:
def threshold_sweep(y_true, pred_label, pred_conf, thresholds):
    rows = []
    for thr in thresholds:
        pred = pred_label.copy()
        pred[pred_conf < thr] = "Other"
        rows.append({
            "threshold": thr,
            "accuracy": accuracy_score(y_true, pred),
            "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
            "coverage_(not_Other)": float((pred != "Other").mean())
        })
    return pd.DataFrame(rows)

if "department" in df_profiles.columns:
    y_true_p = df_profiles["department"].astype(str).values
    thresholds = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

    df_thr = threshold_sweep(
        y_true=y_true_p,
        pred_label=df_profiles["pred_department_raw"].astype(object).values,
        pred_conf=df_profiles["pred_conf"].values,
        thresholds=thresholds
    )
    df_thr
else:
    print("No ground-truth 'department' in df_profiles. Threshold sweep is not possible (accuracy needs labels).")


In [24]:
if "department" in df_profiles.columns:
    print("=== PROFILES CLASSIFICATION REPORT (With Thresholding) ===")
    print(classification_report(df_profiles["department"].astype(str).values,
                                df_profiles["pred_department"].values,
                                digits=3))


=== PROFILES CLASSIFICATION REPORT (With Thresholding) ===
                        precision    recall  f1-score   support

        Administrative      0.222     0.262     0.240        84
  Business Development      0.389     0.474     0.428        78
            Consulting      0.895     0.569     0.696       195
      Customer Support      0.000     0.000     0.000        48
       Human Resources      0.000     0.000     0.000        69
Information Technology      0.309     0.848     0.453       309
             Marketing      0.587     0.481     0.529       133
                 Other      0.661     0.542     0.595      1235
    Project Management      0.849     0.618     0.716       173
            Purchasing      0.000     0.000     0.000        72
                 Sales      0.856     0.790     0.822       219

              accuracy                          0.553      2615
             macro avg      0.434     0.417     0.407      2615
          weighted avg      0.592     0.553

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#2.1 Removing Other

The relative high distribution of "other" in the profiles dataset leads to a class imbalance - by removing "other" out for performance reasons we can evaluate our models performance focusing on predicting the right labels. Without "other" we can also remove the threshold function, which leads to a low performance of our model for rarer classes.

In [25]:
from sklearn.metrics import classification_report, accuracy_score, f1_score, balanced_accuracy_score

# IMPORTANT: assumes df_profiles has ground-truth column "department"
df_eval = df_profiles.copy()

# Remove rows where the true label is "Other"
df_eval_no_other = df_eval[df_eval["department"].astype(str) != "Other"].copy()

print("Profiles total:", len(df_eval))
print("Profiles without true 'Other':", len(df_eval_no_other))
print(df_eval_no_other["department"].value_counts())


Profiles total: 2615
Profiles without true 'Other': 1380
department
Information Technology    309
Sales                     219
Consulting                195
Project Management        173
Marketing                 133
Administrative             84
Business Development       78
Purchasing                 72
Human Resources            69
Customer Support           48
Name: count, dtype: int64


#2.1.1 Evaluation for fine-tuned model without "Other"

In [26]:
y_pred_raw = df_eval_no_other["pred_department_raw"].astype(str).values  # raw argmax
y_true_filtered = df_eval_no_other["department"].astype(str).values # Aligned true labels

print("=== PROFILES OOS (TRUE 'Other' REMOVED) — NO THRESHOLD (RAW) ===")
print("Accuracy:", accuracy_score(y_true_filtered, y_pred_raw))
print("Balanced accuracy:", balanced_accuracy_score(y_true_filtered, y_pred_raw))
print("Macro F1:", f1_score(y_true_filtered, y_pred_raw, average="macro", zero_division=0))
print(classification_report(y_true_filtered, y_pred_raw, digits=3, zero_division=0))

=== PROFILES OOS (TRUE 'Other' REMOVED) — NO THRESHOLD (RAW) ===
Accuracy: 0.6833333333333333
Balanced accuracy: 0.6089795495530835
Macro F1: 0.5966863188259318
                        precision    recall  f1-score   support

        Administrative      0.507     0.452     0.478        84
  Business Development      0.481     0.474     0.477        78
            Consulting      0.904     0.626     0.739       195
      Customer Support      0.818     0.375     0.514        48
       Human Resources      1.000     0.696     0.821        69
Information Technology      0.544     0.883     0.673       309
             Marketing      0.793     0.549     0.649       133
                 Other      0.000     0.000     0.000         0
    Project Management      0.625     0.694     0.658       173
            Purchasing      1.000     0.542     0.703        72
                 Sales      0.911     0.799     0.852       219

              accuracy                          0.683      1380
     

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


#2.1.2 Testing Baseline Model on OOS without Other

In the following we will test our non fine-tuned baseline model to observe how it performs under the same conditions as our fine-tuned one

In [27]:

X_profiles_emb = embedder.encode(
    df_profiles["position"].astype(str).tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

pred_profiles_ids = clf.predict(X_profiles_emb)
df_profiles["pred_department_lr_raw"] = le.inverse_transform(pred_profiles_ids)


Batches:   0%|          | 0/82 [00:00<?, ?it/s]

In [28]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report

df_eval_no_other = df_profiles[df_profiles["department"].astype(str) != "Other"].copy()

y_true = df_eval_no_other["department"].astype(str).values
y_pred = df_eval_no_other["pred_department_lr_raw"].astype(str).values

print("=== PROFILES OOS (TRUE 'Other' REMOVED) — BASELINE (JobBERT embeddings + LogReg), NO THRESHOLD ===")
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average="macro", zero_division=0))
print(classification_report(y_true, y_pred, digits=3, zero_division=0))


=== PROFILES OOS (TRUE 'Other' REMOVED) — BASELINE (JobBERT embeddings + LogReg), NO THRESHOLD ===
Accuracy: 0.7028985507246377
Balanced accuracy: 0.6879685247141766
Macro F1: 0.631922303804158
                        precision    recall  f1-score   support

        Administrative      0.570     0.631     0.599        84
  Business Development      0.429     0.654     0.518        78
            Consulting      0.754     0.708     0.730       195
      Customer Support      0.548     0.479     0.511        48
       Human Resources      0.902     0.797     0.846        69
Information Technology      0.774     0.654     0.709       309
             Marketing      0.754     0.647     0.696       133
                 Other      0.000     0.000     0.000         0
    Project Management      0.567     0.734     0.640       173
            Purchasing      0.982     0.750     0.850        72
                 Sales      0.879     0.826     0.852       219

              accuracy              

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


# Conclusion:

In an out-of-sample evaluation on profile data without the “Other” class, a baseline using frozen JobBERT embeddings with a logistic regression classifier slightly outperforms the fine-tuned model in terms of macro-F1.
This behavior is expected under distribution shift, as the baseline learns smoother decision boundaries, while the fine-tuned model is optimized for in-distribution performance and exhibits sharper class boundaries.

The fine-tuned model, however, enables confidence-based abstention and achieves near-perfect performance in a closed-world setting, highlighting the trade-off between robustness and precision control.


